# types_python_dateutil Package Testing

`types-python-dateutil` is a stub-only distribution: it installs `dateutil-stubs/*.pyi` type stubs and no importable module. The first cell checks that the stub files are installed; the examples use the `dateutil` APIs those stubs describe, with type annotations.

In [ ]:
# Stub verification: types-python-dateutil installs dateutil-stubs/*.pyi (no importable module).
import importlib.metadata

dist = importlib.metadata.distribution('types-python-dateutil')
print(f"types-python-dateutil version: {dist.version}")

pyi_files = [f for f in (dist.files or []) if str(f).endswith('.pyi')]
print(f"stub files: {len(pyi_files)}")
for name in sorted(str(f) for f in pyi_files)[:5]:
    print(f"  {name}")

assert pyi_files, "no .pyi stub files found"
assert any('dateutil-stubs' in str(f) for f in pyi_files), "dateutil-stubs dir missing"
assert all(f.locate().exists() for f in pyi_files[:3]), "stub files missing on disk"
print("OK — dateutil-stubs installed and present on disk")

In [ ]:
# Example 1: type-safe parsing with dateutil.parser (APIs covered by dateutil-stubs).
from datetime import datetime
from dateutil import parser

dt: datetime = parser.parse("2026-06-08T14:30:00+02:00")
print(f"parsed: {dt!r}")
assert (dt.year, dt.month, dt.day) == (2026, 6, 8)
assert dt.utcoffset() is not None

fuzzy: datetime = parser.parse("The launch happened on 8 June 2026 at 14:30", fuzzy=True)
print(f"fuzzy: {fuzzy}")
assert (fuzzy.year, fuzzy.month, fuzzy.day) == (2026, 6, 8)
print("dateutil.parser typed usage OK")

In [ ]:
# Example 2: relativedelta arithmetic with type annotations.
from datetime import date
from dateutil.relativedelta import relativedelta

start: date = date(2026, 1, 31)
one_month: relativedelta = relativedelta(months=+1)
print(f"{start} + 1 month = {start + one_month}")
assert start + one_month == date(2026, 2, 28)  # clamps to end of February

diff: relativedelta = relativedelta(date(2026, 6, 8), date(2020, 1, 1))
print(f"diff = {diff.years}y {diff.months}m {diff.days}d")
assert diff.years == 6
print("relativedelta typed usage OK")

In [ ]:
# Example 3: rrule recurrence generation with type annotations.
from datetime import datetime
from dateutil.rrule import MO, WEEKLY, rrule

dates: list[datetime] = list(rrule(WEEKLY, byweekday=MO, dtstart=datetime(2026, 6, 1), count=4))
for d in dates:
    print(d.date(), d.strftime('%A'))

assert len(dates) == 4
assert all(d.weekday() == 0 for d in dates)
assert dates[0] == datetime(2026, 6, 1)
print("rrule typed usage OK")

In [ ]:
# Example 4: IANA timezone handling with dateutil.tz.
from datetime import datetime, timedelta
from dateutil import tz

nyc = tz.gettz("America/New_York")
assert nyc is not None
winter = datetime(2026, 1, 15, 12, 0, tzinfo=nyc)
summer = datetime(2026, 7, 15, 12, 0, tzinfo=nyc)
print(f"winter offset: {winter.utcoffset()}, summer offset: {summer.utcoffset()}")
assert winter.utcoffset() == timedelta(hours=-5)  # EST
assert summer.utcoffset() == timedelta(hours=-4)  # EDT

assert datetime(2026, 6, 8, tzinfo=tz.UTC).utcoffset() == timedelta(0)
print("dateutil.tz typed usage OK")

In [ ]:
# Example 5: isoparse round-trip + easter calculation.
from datetime import date, datetime
from dateutil.easter import easter
from dateutil.parser import isoparse

stamp = "2026-06-08T14:30:00+00:00"
dt: datetime = isoparse(stamp)
assert dt.isoformat() == stamp
print(f"isoparse round-trip OK: {dt.isoformat()}")

e: date = easter(2026)
print(f"Easter 2026: {e}")
assert e == date(2026, 4, 5)
assert e.weekday() == 6  # always a Sunday
print("isoparse + easter typed usage OK")